# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
subprocess.check_call(['apt-get', '-qq', 'update'])
subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — base + GPU extras (torch CUDA, decord, av)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
# Kaggle's base image already includes a CUDA torch; install the GPU
# extras NOT pinned to a specific torch wheel (skip torch line to
# avoid clobbering Kaggle's preinstalled CUDA torch).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'decord==0.6.0', 'av==12.3.0'])
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())

In [ ]:
# 3) Bootstrap the Firebase credential from the attached Kaggle Dataset.
#
# Why a Dataset instead of Kaggle Secrets:
#   Kaggle's Notebook → Add-ons → Secrets panel detaches every attached
#   secret on each new notebook version. Since GitHub Actions does
#   'kaggle kernels push' on every dashboard Run click, that meant
#   re-attaching secrets manually after every wake-up. Untenable.
#
#   Datasets are referenced in kernel-metadata.json's dataset_sources
#   field, so they're auto-attached on every push. Once-set, always-on.
#
# Setup (one-time):
#   1. Bundle your Firebase service-account JSON as a PRIVATE Kaggle
#      Dataset named  <kaggle_username>/yt-agent-secrets  containing
#      one file: firebase-sa.json.
#   2. The kernel-metadata.json already lists this dataset as a source,
#      so every push from GitHub Actions auto-attaches it.
#   3. To rotate the Firebase credential, replace firebase-sa.json in
#      that dataset (kaggle datasets version --dir-mode zip -p <path>
#      -m "rotate creds"). The next worker boot picks up the new
#      version automatically.
#
# Everything else (R2_*, SFTP_*, NIM, Groq, etc.) flows in from Firestore
# at server startup via backend/keys_sync.pull_into_env().
import os, pathlib, base64, json

SECRETS_DIR = pathlib.Path('/kaggle/input/yt-agent-secrets')
SA_FILE     = SECRETS_DIR / 'firebase-sa.json'

if not SA_FILE.exists():
    raise RuntimeError(
        f'Firebase service-account JSON not found at {SA_FILE}. '
        f'Verify kernel-metadata.json lists "<your-username>/yt-agent-secrets" '
        f'under dataset_sources AND the dataset contains firebase-sa.json.'
    )

sa_raw = SA_FILE.read_text(encoding='utf-8')
# Sanity-check it's valid JSON before propagating downstream.
json.loads(sa_raw)
os.environ['GOOGLE_APPLICATION_CREDENTIALS_JSON'] = sa_raw

# Identity. backend/registry auto-detects the actual GPU model via
# nvidia-smi and appends it, so the dashboard shows e.g.
# "kaggle · Tesla P100-PCIE-16GB" — accurate regardless of which
# GPU Kaggle assigned this session.
os.environ['INSTANCE_TIER']  = 'gpu'
os.environ['INSTANCE_LABEL'] = 'kaggle'
# Self-terminate after 10 min of idle so we don't waste Kaggle GPU hours.
os.environ['KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS'] = '600'

print(f'Bootstrap: loaded Firebase creds from {SA_FILE} ({len(sa_raw)} chars).')
print('Everything else will be pulled from Firestore at server startup.')

In [ ]:
# 4) Cloudflare quick tunnel — same pattern as Colab cell 6.
import os, subprocess, time, re
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.check_call([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared',
    ])
    subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
tunnel_log = '/tmp/cloudflared.log'
subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
     '--logfile', tunnel_log, '--loglevel', 'info'],
    stdout=open(tunnel_log + '.stdout', 'w'), stderr=subprocess.STDOUT,
)
url = None
for _ in range(40):
    time.sleep(0.5)
    if not os.path.exists(tunnel_log):
        continue
    with open(tunnel_log) as f:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
        if m:
            url = m.group(0); break
if not url:
    raise RuntimeError('cloudflared did not produce a URL')
os.environ['PUBLIC_BACKEND_URL'] = url
print('Public backend URL:', url)

In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])